In [1]:
import sys

import polars as pl
import torch


from modeling_module.data_loader.MultiPartDataModule import MultiPartDataModule
from modeling_module.data_loader.MultiPartExoDataModule import MultiPartExoDataModule

'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

save_dir = DIR + 'fit/model_validation'


In [2]:
from modeling_module.utils.date_util import DateUtil

ETT1 = (pl.read_csv(DIR + 'csv/ETTm1.csv'))
ETT1_sample = (ETT1
               .select(['date', 'HUFL'])
               .with_columns(pl.lit('A').alias('unique_id'))
               .with_columns(pl.col('date').map_elements(DateUtil.parse_to_yyyymmddhh).alias('date'))
               )
ETT1_sample

/var/folders/py/xgf_87rd5nz9rsbc143wp9qc0000gn/T/ipykernel_72661/2245021606.py:7: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(pl.col('date').map_elements(DateUtil.parse_to_yyyymmddhh).alias('date'))


date,HUFL,unique_id
i64,f64,str
2016070100,5.827,"""A"""
2016070100,5.76,"""A"""
2016070100,5.76,"""A"""
2016070100,5.76,"""A"""
2016070101,5.693,"""A"""
…,…,…
2018062618,9.31,"""A"""
2018062619,10.114,"""A"""
2018062619,10.784,"""A"""


In [ ]:
# import matplotlib.pyplot as plt
# plt.figure(figsize = (18, 6))
# plt.plot(ETT1['date'], ETT1['HUFL'])
# plt.xticks(rotation = 90, fontsize = 8)
# plt.title('ETT Dataset - Weekly Mean OT', fontsize = 16)
# plt.xlabel('Year-Week', fontsize = 12)
# plt.ylabel('Mean OT', fontsize = 12)
# plt.grid(True, linestyle = '--', alpha = 0.4)
#
# plt.tight_layout()
# plt.show()

In [3]:
plan_yyyyww = 201801
lookback = 72
horizon = 24

data_module = MultiPartExoDataModule(
    ETT1_sample,
    part_col = 'unique_id',
    date_col = 'date',
    qty_col = 'HUFL',
    lookback = lookback,
    horizon = horizon,
    freq = 'hourly'
)

train_loader = data_module.get_train_loader()
val_loader = data_module.get_val_loader()

In [4]:
print("len(train_dataset) =", len(data_module.train_dataset))
print("len(val_dataset)   =", len(data_module.val_dataset))

train_loader = data_module.get_train_loader()
val_loader   = data_module.get_val_loader()

print("len(train_loader) =", len(train_loader))
print("len(val_loader)   =", len(val_loader))


len(train_dataset) = 55668
len(val_dataset)   = 13917
len(train_loader) = 1739
len(val_loader)   = 435


In [ ]:
from modeling_module.training.model_trainers.total_train import run_total_train_monthly, run_total_train_weekly

model_dict = run_total_train_weekly(
    train_loader,
    val_loader,
    lookback = lookback,
    horizon = horizon,
    save_dir = save_dir
)

In [ ]:
from modeling_module.utils.checkpoint import load_model_dict
# Load
from modeling_module.models.model_builder import (
    build_patch_mixer_quantile,
    build_patchTST_base, build_patchTST_quantile, build_patch_mixer_base, build_titan_base, build_titan_lmm,
    build_titan_seq2seq,
)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

builders = {
    # f"weekly_PatchMixerBase_L{lookback}_H{horizon}": build_patch_mixer_base,
    # f"weekly_PatchMixerQuantile_L{lookback}_H{horizon}": build_patch_mixer_quantile,
    f"weekly_TitanBase_L{lookback}_H{horizon}": build_titan_base,
    f"weekly_TitanLMM_L{lookback}_H{horizon}": build_titan_lmm,
    f"weekly_TitanSeq2Seq_L{lookback}_H{horizon}": build_titan_seq2seq,
    # f"weekly_PatchTSTBase_L{lookback}_H{horizon}": build_patchTST_base,
    # f"weekly_PatchTSTQuantile_L{lookback}_H{horizon}": build_patchTST_quantile,
}
loaded = load_model_dict(save_dir, builders, device = device)

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib, modeling_module.utils.plot_utils as pu
import modeling_module.training.forecaster as fo
importlib.reload(pu)
importlib.reload(fo)

def my_exo_cb(start_idx: int, Hm: int, device="cuda" if torch.cuda.is_available() else "cpu"):
    # exo_dim = 2 (sin, cos)
    return fo.make_calendar_exo(start_idx, Hm, period=54, device=device)

pu.plot_27w(
    models=loaded,           # {"PatchMixer": pm_model, "Titan": ti_model, ...}
    loader=val_loader,       # (xb, yb[, part_ids])
    device="cuda" if torch.cuda.is_available() else "cpu",
    mode="val",              # ← 검증 모드
    max_plots=1,
    out_dir=None,
    show=True,
    future_exo_cb=my_exo_cb
)